<a href="https://colab.research.google.com/github/CMB-i/simple-reflexive-agent-AQI-/blob/main/AQI_simple_reflexive_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import kagglehub
import os


path = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")

print("Dataset path:", path)


PATH_STATION_HOUR = os.path.join(path, "station_hour.csv")
PATH_STATION_DAY = os.path.join(path, "station_day.csv")
PATH_CITY_HOUR = os.path.join(path, "city_hour.csv")
PATH_CITY_DAY = os.path.join(path, "city_day.csv")
PATH_STATIONS = os.path.join(path, "stations.csv")

STATIONS = ["KL007", "KL008"]  # choose stations
MIN_VALID_POLLUTANTS = 3       # CPCB-like rule: need >=3 pollutants

Using Colab cache for faster access to the 'air-quality-data-in-india' dataset.
Dataset path: /kaggle/input/air-quality-data-in-india


In [3]:
BREAKPOINTS = {
    "PM2.5": [
        (0, 30, 0, 50),
        (31, 60, 51, 100),
        (61, 90, 101, 200),
        (91, 120, 201, 300),
        (121, 250, 301, 400),
        (251, 350, 401, 500),
    ],
    "PM10": [
        (0, 50, 0, 50),
        (51, 100, 51, 100),
        (101, 250, 101, 200),
        (251, 350, 201, 300),
        (351, 430, 301, 400),
        (431, 600, 401, 500),
    ],
    "SO2": [
        (0, 40, 0, 50),
        (41, 80, 51, 100),
        (81, 380, 101, 200),
        (381, 800, 201, 300),
        (801, 1600, 301, 400),
        (1601, 2000, 401, 500),
    ],
    "NOx": [  # NOTE: many standards use NO2; dataset has NOx, tutorial used NOx
        (0, 40, 0, 50),
        (41, 80, 51, 100),
        (81, 180, 101, 200),
        (181, 280, 201, 300),
        (281, 400, 301, 400),
        (401, 500, 401, 500),
    ],
    "NH3": [
        (0, 200, 0, 50),
        (201, 400, 51, 100),
        (401, 800, 101, 200),
        (801, 1200, 201, 300),
        (1201, 1800, 301, 400),
        (1801, 2400, 401, 500),
    ],
    "CO": [  # CO in mg/m3 (as in dataset)
        (0, 1, 0, 50),
        (1.1, 2, 51, 100),
        (2.1, 10, 101, 200),
        (10.1, 17, 201, 300),
        (17.1, 34, 301, 400),
        (34.1, 50, 401, 500),
    ],
    "O3": [
        (0, 50, 0, 50),
        (51, 100, 51, 100),
        (101, 168, 101, 200),
        (169, 208, 201, 300),
        (209, 748, 301, 400),
        (749, 1000, 401, 500),
    ],
}


In [4]:
def compute_subindex(conc, bps):
    """Compute AQI subindex using linear interpolation across breakpoints."""
    if pd.isna(conc):
        return np.nan

    conc = float(conc)
    for c_lo, c_hi, i_lo, i_hi in bps:
        if c_lo <= conc <= c_hi:
            # Linear interpolation:
            return (i_hi - i_lo) / (c_hi - c_lo) * (conc - c_lo) + i_lo

    # If beyond highest breakpoint -> clamp to 500
    if conc > bps[-1][1]:
        return 500.0
    return np.nan

def aqi_bucket(aqi):
    if pd.isna(aqi):
        return np.nan
    aqi = float(aqi)
    if aqi <= 50: return "Good"
    if aqi <= 100: return "Satisfactory"
    if aqi <= 200: return "Moderate"
    if aqi <= 300: return "Poor"
    if aqi <= 400: return "Very Poor"
    return "Severe"

HEALTH_MSG = {
    "Good": "Air quality is good Enjoy normal outdoor activities.",
    "Satisfactory": "Minor breathing discomfort to sensitive people.",
    "Moderate": "Breathing discomfort to people with lung/heart disease, children, older adults.",
    "Poor": "Breathing discomfort to most people on prolonged exposure.",
    "Very Poor": "Respiratory illness on prolonged exposure. Avoid outdoor exertion.",
    "Severe": "Serious health effects. Stay indoors, wear masks if necessary"
}

def health_message(bucket):
    if pd.isna(bucket):
        return np.nan
    return HEALTH_MSG.get(bucket, "")

In [5]:
# Simple Reflex Agent

class AQIReflexAgent:
    """
    Simple Reflex Agent:
    Percept  -> pollutant readings (rolling avg/max from file)
    Rules    -> compute subindices + pick max
    Action   -> output AQI + bucket + dominant pollutant + message
    """
    def __init__(self, breakpoints, min_valid_pollutants=3):
        self.breakpoints = breakpoints
        self.min_valid_pollutants = min_valid_pollutants
        self.pollutants = list(breakpoints.keys())

    def preprocess(self, df):
        """Compute required rolling stats based on dataset frequency."""
        df = df.copy()
        df["Datetime"] = pd.to_datetime(df["Datetime"])
        df.sort_values(["StationId", "Datetime"], inplace=True)

        # Rolling averages / max windows
        df["PM10_24hr_avg"]  = df.groupby("StationId")["PM10"].rolling(24, min_periods=16).mean().reset_index(level=0, drop=True)
        df["PM2.5_24hr_avg"] = df.groupby("StationId")["PM2.5"].rolling(24, min_periods=16).mean().reset_index(level=0, drop=True)
        df["SO2_24hr_avg"]   = df.groupby("StationId")["SO2"].rolling(24, min_periods=16).mean().reset_index(level=0, drop=True)
        df["NOx_24hr_avg"]   = df.groupby("StationId")["NOx"].rolling(24, min_periods=16).mean().reset_index(level=0, drop=True)
        df["NH3_24hr_avg"]   = df.groupby("StationId")["NH3"].rolling(24, min_periods=16).mean().reset_index(level=0, drop=True)

        df["CO_8hr_max"]     = df.groupby("StationId")["CO"].rolling(8, min_periods=1).max().reset_index(level=0, drop=True)
        df["O3_8hr_max"]     = df.groupby("StationId")["O3"].rolling(8, min_periods=1).max().reset_index(level=0, drop=True)

        return df

    def perceive(self, row):
        """Extract the 'sensor percept' vector from a row."""
        return {
            "PM2.5": row.get("PM2.5_24hr_avg", np.nan),
            "PM10": row.get("PM10_24hr_avg", np.nan),
            "SO2":  row.get("SO2_24hr_avg", np.nan),
            "NOx":  row.get("NOx_24hr_avg", np.nan),
            "NH3":  row.get("NH3_24hr_avg", np.nan),
            "CO":   row.get("CO_8hr_max", np.nan),
            "O3":   row.get("O3_8hr_max", np.nan),
        }

    def act(self, percept):
        """Rule-based action: compute AQI as max subindex."""
        sub = {}
        for p, val in percept.items():
            sub[p] = compute_subindex(val, self.breakpoints[p])

        # Count valid pollutant subindices
        valid = [v for v in sub.values() if not pd.isna(v) and v > 0]
        if len(valid) < self.min_valid_pollutants:
            return {
                "AQI": np.nan,
                "Bucket": np.nan,
                "DominantPollutant": np.nan,
                "Message": np.nan,
                "SubIndices": sub
            }

        dominant = max(sub, key=lambda k: (-np.nan_to_num(sub[k], nan=-1), k))
        aqi = round(np.nanmax(list(sub.values())))

        bucket = aqi_bucket(aqi)
        return {
            "AQI": aqi,
            "Bucket": bucket,
            "DominantPollutant": dominant,
            "Message": health_message(bucket),
            "SubIndices": sub
        }

    def run(self, df):
        """Run agent over full dataframe."""
        df = self.preprocess(df)

        outputs = []
        for _, row in df.iterrows():
            percept = self.perceive(row)
            out = self.act(percept)
            outputs.append(out)

        out_df = pd.DataFrame(outputs)
        # Expand subindices into columns
        sub_df = pd.json_normalize(out_df["SubIndices"])
        sub_df.columns = [f"{c}_SubIndex" for c in sub_df.columns]
        out_df = pd.concat([out_df.drop(columns=["SubIndices"]), sub_df], axis=1)

        return pd.concat([df.reset_index(drop=True), out_df], axis=1)

In [6]:
import numpy as np
import pandas as pd
df = pd.read_csv(PATH_STATION_HOUR, parse_dates=["Datetime"], low_memory=False)
stations = pd.read_csv(PATH_STATIONS)

df = df.merge(stations, on="StationId", how="left")
df = df[df["StationId"].isin(STATIONS)].copy()

agent = AQIReflexAgent(BREAKPOINTS, min_valid_pollutants=MIN_VALID_POLLUTANTS)
df_out = agent.run(df)


df_out = df_out.rename(columns={"AQI": "AQI_calc", "Bucket": "Bucket_calc"})

df_out = df_out.loc[:, ~df_out.columns.duplicated()]


sub_cols = [c for c in df_out.columns if c.endswith("_SubIndex")]
df_out["valid_pollutants"] = df_out[sub_cols].apply(lambda r: np.sum((r > 0) & (~pd.isna(r))), axis=1)

print("Total rows:", len(df_out))
print("Computed AQI rows:", df_out["AQI_calc"].notna().sum())
print("NaN AQI rows:", df_out["AQI_calc"].isna().sum())

print("\nTop Dominant Pollutants:")
print(df_out.loc[df_out["AQI_calc"].notna(), "DominantPollutant"].value_counts().head(10))

print("\nSample computed rows:")
print(df_out.loc[df_out["AQI_calc"].notna(),
                 ["StationId","City","Datetime","AQI_calc","Bucket_calc","DominantPollutant","valid_pollutants"]
                ].head(20))

print("\nAQI Bucket Counts (Agent):")
print(df_out["Bucket_calc"].value_counts(dropna=True))

Total rows: 29971
Computed AQI rows: 27265
NaN AQI rows: 2706

Top Dominant Pollutants:
DominantPollutant
NH3      18644
PM2.5     2875
CO        2194
SO2       2106
PM10       977
O3         435
NOx         33
Name: count, dtype: int64

Sample computed rows:
   StationId                City            Datetime  AQI_calc   Bucket_calc  \
15     KL007  Thiruvananthapuram 2020-02-14 08:00:00      66.0  Satisfactory   
16     KL007  Thiruvananthapuram 2020-02-14 09:00:00      73.0  Satisfactory   
17     KL007  Thiruvananthapuram 2020-02-14 10:00:00      73.0  Satisfactory   
18     KL007  Thiruvananthapuram 2020-02-14 11:00:00      81.0  Satisfactory   
19     KL007  Thiruvananthapuram 2020-02-14 12:00:00      88.0  Satisfactory   
20     KL007  Thiruvananthapuram 2020-02-14 13:00:00     100.0  Satisfactory   
21     KL007  Thiruvananthapuram 2020-02-14 14:00:00     100.0  Satisfactory   
22     KL007  Thiruvananthapuram 2020-02-14 15:00:00     105.0      Moderate   
23     KL007  Thiruv